# Kev-0.8B domain fine-tuning (Colab or Kaggle T4)

Runs one session of the Milestone 4 plan (`configs/train.yaml`, `sessions`; protocol in
`docs/specs/kev_training_protocol.md`). One notebook serves both platforms:

| | Colab (free T4) | Kaggle (2 x T4) |
| --- | --- | --- |
| Data | Drive folder `Group4/data` | Private dataset `group4-kev-data` (same files) |
| Output | Drive folder `Group4/checkpoints/` | `/kaggle/working/checkpoints/` (notebook output) |
| Runs at once | 1 | 2, one queue per GPU |
| Unattended | No: keep the tab open | Yes: "Save Version", "Save & Run All" |

Sessions: `kaggle_1`, `kaggle_2`, `kaggle_3`, `colab_1`. Set `SMOKE = True` the first time on each
platform (one smoke run per GPU, 16 optimizer steps on 32 records), then `SMOKE = False` with the
session name. Each run is saved the moment it finishes; a failed run is saved as
`<run id>.failed-<time>` and never overwrites a finished run. Every run uses fp32 (the T4 has no
bf16 and `kev.train` has no fp16). A LoRA run cannot resume after a disconnect: rerun it.
One Colab and one Kaggle account per person.

In [ ]:
# ---- parameters ----
PLATFORM = "kaggle"        # "kaggle" or "colab"
SMOKE = True               # True: the smoke run(s) of this platform; False: the session below
SESSION = "kaggle_1"       # kaggle_1, kaggle_2, kaggle_3 or colab_1 (configs/train.yaml, sessions)
KEV_COMMIT = "2074cc44256200bb37540103fa4572901faa10f3"
KAGGLE_DATASET = "group4-kev-data"   # the private dataset holding the Group4/data files
MIN_FREE_GB = 25           # scratch space needed for the Kev environment and the model cache

In [ ]:
import hashlib, json, os, pathlib, platform, shlex, shutil, subprocess, sys, time
from datetime import datetime, timezone

def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = pathlib.Path("/content/kev_work")
    DATA = pathlib.Path("/content/drive/MyDrive/Group4/data")
    OUT = pathlib.Path("/content/drive/MyDrive/Group4/checkpoints")
elif PLATFORM == "kaggle":
    WORK = pathlib.Path("/tmp/kev_work")              # local disk; not saved with the output
    found = sorted(pathlib.Path("/kaggle/input").rglob("kev_plan.json"))
    assert found, f"attach the dataset {KAGGLE_DATASET} (Add Input > Your Datasets)"
    DATA = found[0].parent
    OUT = pathlib.Path("/kaggle/working/checkpoints") # saved as the notebook output
else:
    raise ValueError(PLATFORM)
LOGS = OUT / "_logs"                                  # live logs, on persistent storage
for d in (WORK, OUT, LOGS):
    d.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(WORK).free / 1e9
print(f"data: {DATA}\nfree scratch space: {free_gb:.0f} GB")
assert free_gb >= MIN_FREE_GB, f"only {free_gb:.0f} GB free in {WORK}"
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip().splitlines()
print("GPUs:", gpus)
assert gpus, "no GPU: set the accelerator to T4 (Colab: Runtime > Change runtime type; Kaggle: Settings > Accelerator)"

In [ ]:
# The runs of this session: plan file, platform, data hashes, and no finished run overwritten.
plan_file = json.loads((DATA / (f"kev_smoke_{PLATFORM}.json" if SMOKE else "kev_plan.json")).read_text())
specs = {r["run_id"]: r for r in plan_file["runs"]}
meta = plan_file["meta"]
print("plan made from repository commit", meta["repo_commit"], "(working tree dirty)" if meta["repo_dirty"] else "")
queues = [[run_id] for run_id in specs] if SMOKE else meta["sessions"][SESSION]
assert len(queues) <= len(gpus), f"{len(queues)} GPU queues but {len(gpus)} GPU(s)"
for queue in queues:
    for run_id in queue:
        assert specs[run_id]["platform"] == PLATFORM, f"{run_id} is planned for {specs[run_id]['platform']}"
        assert not (OUT / run_id).exists(), f"{OUT / run_id} exists: a finished run is never overwritten"

def local_data(run):
    path = DATA / "kev_domain" / pathlib.Path(run["data"]).name
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == run["data_sha256"], f"hash mismatch for {path}"
    return path, digest

for queue in queues:
    for run_id in queue:
        print(run_id, local_data(specs[run_id])[0], "ok")
print("GPU queues:", queues)

In [ ]:
# Kev at the pinned commit (clean tree), installed from its lock file with uv: Python 3.13, no dev
# dependencies, the same package versions on every platform. Needs internet (Kaggle: Settings > Internet on).
os.chdir(WORK)
if not (WORK / "kev").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/jaredpalmer/kev.git"], check=True)
os.chdir(WORK / "kev")
subprocess.run(["git", "checkout", "-q", KEV_COMMIT], check=True)
dirty = subprocess.run(["git", "status", "--porcelain", "--untracked-files=no"], capture_output=True, text=True).stdout
assert not dirty.strip(), f"the Kev clone has local changes:\n{dirty}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--python", "3.13", "--frozen", "--no-dev", "-q"], check=True)
PY = str(WORK / "kev" / ".venv" / "bin" / "python")
probe = ("import json, sys, torch, transformers, peft; "
         "assert torch.cuda.is_available(), 'PyTorch cannot use CUDA'; "
         "n = torch.cuda.device_count(); "
         "[(torch.ones(256, 256, device=f'cuda:{i}') @ torch.ones(256, 256, device=f'cuda:{i}')).sum().item() for i in range(n)]; "
         "print(json.dumps({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.version.cuda, "
         "'transformers': transformers.__version__, 'peft': peft.__version__, 'cuda_devices': n, "
         "'device_names': [torch.cuda.get_device_name(i) for i in range(n)]}))")
versions = json.loads(subprocess.run([PY, "-c", probe], capture_output=True, text=True, check=True).stdout)
print(versions)
assert versions["cuda_devices"] >= len(queues), "PyTorch sees fewer GPUs than queues"
freeze = subprocess.run(["uv", "pip", "freeze", "--python", PY], capture_output=True, text=True).stdout
uv_lock_sha256 = hashlib.sha256((WORK / "kev" / "uv.lock").read_bytes()).hexdigest()

In [ ]:
# Download the base model and the pinned warm start once, before the runs start in parallel.
first = specs[queues[0][0]]
init_repo, _, init_rev = meta["init_from"].partition("@")
pre = (f"from huggingface_hub import snapshot_download as s; "
       f"s('{first['base']}', revision='{first['base_revision']}'); "
       f"s('{init_repo}', revision='{init_rev}')")
subprocess.run([PY, "-c", pre], check=True)

In [ ]:
# Train: one queue per GPU, run in order. Each run is validated and saved as soon as it exits.
REQUIRED = ("adapter_model.safetensors", "adapter_config.json", "head.pt", "training_metrics.json")
results = {}

def start(run_id, gpu):
    run = specs[run_id]
    run_dir = WORK / "kev" / "runs" / run_id
    if run_dir.exists():
        shutil.rmtree(run_dir)                      # scratch only; kev.train refuses an existing --out
    data_path, digest = local_data(run)
    command = run["command"].replace(f"--data {run['data']}", f"--data {shlex.quote(str(data_path))}")
    command = command.replace(f"--out runs/{run_id}", f"--out {shlex.quote(str(run_dir))}")
    command = command.replace("python -m kev.train", f"{PY} -m kev.train")
    log_path = LOGS / f"{run_id}.train.log"
    record = {"run_id": run_id, "platform": PLATFORM, "session": "smoke" if SMOKE else SESSION,
              "gpu": gpus[gpu], "gpu_index": gpu, "parallel_queues": len(queues), "kev_commit": KEV_COMMIT,
              "uv_lock_sha256": uv_lock_sha256, **versions, "os": platform.platform(),
              "data_repo_path": run["data"], "data_sha256": digest, "init_from": run["init_from"],
              "plan_meta": meta, "command": command, "started_utc": now()}
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    log = open(log_path, "w")
    print(f"[GPU {gpu}] start {run_id}: {command}", flush=True)
    proc = subprocess.Popen(shlex.split(command), stdout=log, stderr=subprocess.STDOUT, env=env, cwd=WORK / "kev")
    return {"run_id": run_id, "run_dir": run_dir, "log": log, "log_path": log_path, "record": record, "proc": proc}

def finish(job):
    job["log"].close()
    run_id, run_dir, record = job["run_id"], job["run_dir"], job["record"]
    record["ended_utc"] = now()
    record["exit_code"] = job["proc"].returncode
    missing = [f for f in REQUIRED if not (run_dir / f).exists()]
    ok = record["exit_code"] == 0 and not missing
    target = OUT / run_id if ok else OUT / f"{run_id}.failed-{datetime.now(timezone.utc):%Y%m%dT%H%M%S}"
    if ok:
        shutil.copytree(run_dir, target)            # target does not exist (checked above)
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy(job["log_path"], target / "train.log")
    (target / "environment.json").write_text(json.dumps(record, indent=2) + "\n")
    (target / "pip_freeze.txt").write_text(freeze)
    results[run_id] = {"ok": ok, "exit_code": record["exit_code"], "missing": missing, "saved_to": str(target)}
    print(f"{'saved' if ok else 'FAILED'} {run_id} -> {target}", flush=True)
    if ok:
        for name in ("adapter_model.safetensors", "head.pt"):
            print("  ", name, hashlib.sha256((target / name).read_bytes()).hexdigest())
        m = json.loads((target / "training_metrics.json").read_text())
        print("  ", {k: m.get(k) for k in ("wall_seconds", "records_seen", "optimizer_steps", "peak_device_bytes", "grad_norm")})

pending = [list(q) for q in queues]
running = {gpu: start(pending[gpu].pop(0), gpu) for gpu in range(len(pending)) if pending[gpu]}
while running:
    time.sleep(30)
    for gpu, job in list(running.items()):
        if job["proc"].poll() is None:
            tail = job["log_path"].read_text().strip().splitlines()[-1:]
            print(f"[GPU {gpu}] {job['run_id']}: {tail}", flush=True)
            continue
        finish(job)
        del running[gpu]
        if pending[gpu]:
            running[gpu] = start(pending[gpu].pop(0), gpu)

In [ ]:
# Summary: every run of the session must have finished and been saved.
for run_id, r in results.items():
    print(run_id, "ok" if r["ok"] else f"FAILED (exit {r['exit_code']}, missing {r['missing']})", r["saved_to"])
failed = [k for k, r in results.items() if not r["ok"]]
assert not failed, f"failed runs: {failed} (see their train.log)"

## After the session

- Kaggle: open the saved version, "Output", and download `checkpoints/<run id>` for each run.
- Colab: the runs are already in Drive under `Group4/checkpoints/<run id>`.
- On the machine with the repository, put each run folder in `models/checkpoints/<run id>`, then
  follow `docs/specs/kev_training_protocol.md` Section 4, steps 5 to 8 (score, register, records).